# 08: Proyecto: entorno de desarrollo completo con docker-compose


Es hora de poner todo junto. En esta lección vas a construir un entorno de desarrollo profesional COMPLETO: 

1. un pipeline ETL en Python que extrae datos de un archivo CSV, 
2. los transforma, 
3. los carga en PostgreSQL, 
4. puedes verificar el resultado en pgAdmin.

Todo orquestado con docker-compose: 

- Reproducible. 
- Cualquier persona de tu equipo puede hacer git clone + docker compose up y tener el mismo entorno funcionando en minutos.



## Arquitectura del proyecto


Proyecto: Mini Pipeline ETL en Docker
docker-compose
CSV
ventas_raw.csv
(bind mount)
ETL Python
Lee CSV
Limpia + Transforma
Carga en PostgreSQL
PostgreSQL
Datos limpios
(named volume)
pgAdmin
Verificar
resultados
localhost:5432
localhost:8080
docker compose up → todo arranca automáticamente
git clone + docker compose up = entorno listo en 2 minutos



## Paso 1: Estructura del proyecto


```bash
# Estructura de archivos del proyecto
mi-pipeline-docker/
├── docker-compose.yml      # Orquesta todos los servicios
├── Dockerfile              # Imagen personalizada para el ETL
├── requirements.txt        # Dependencias Python del ETL
├── etl/
│   └── pipeline.py         # El script ETL principal
├── data/
│   └── ventas_raw.csv      # Datos crudos de entrada
└── init-db/
    └── schema.sql          # Esquema de la tabla destino
```

 `docker-compose.yml` → Orquestador: define servicios, puertos, volúmenes y dependencias.
- `Dockerfile` → Receta para construir la imagen del ETL.
- `requirements.txt` → Dependencias Python (psycopg2-binary).
- `etl/pipeline.py` → El script principal que ejecuta Extract → Transform → Load.
- `data/ventas_raw.csv` → Datos de entrada (bind mount al contenedor).
- `init-db/schema.sql` → Se ejecuta automáticamente al primer arranque de PostgreSQL.

Esta estructura es un patrón que reutilizarás en cada proyecto de datos. Memorízala.

## Paso 2: Los datos crudos (CSV)


# Archivo: data/ventas_raw.csv

```csv
id,producto,cantidad,precio_unitario,fecha,cliente,ciudad
1,Laptop Pro,2,1299.99,2025-01-15,Ana García,Madrid
2,Mouse Wireless,,29.99,2025-01-15,Carlos López,Barcelona
3,Teclado Mecánico,1,89.50,15/01/2025,María Fernández,Valencia
4,Monitor 4K,1,549.00,2025-01-16,,Sevilla
5,Laptop Pro,3,1299.99,2025-01-16,Ana García,Madrid
6,USB Hub,-1,19.99,2025-01-17,Pedro Martín,Bilbao
7,Webcam HD,2,79.99,2025-01-17,Laura Sánchez,Madrid
8,Monitor 4K,1,549.00,2025-01-18,Carlos López,Barcelona
```

- Fila 2: cantidad vacía (campo entre comas sin valor) — hay que decidir si descartar o imputar.
- Fila 3: fecha en formato DD/MM/YYYY en vez de YYYY-MM-DD — hay que normalizar.
- Fila 4: cliente vacío — elegimos "Desconocido" como valor por defecto.
- Fila 6: cantidad negativa (-1) — inválido para una venta, hay que descartarla.

Estos errores son 100% realistas. En la vida real, el 80% del trabajo de data engineering es limpiar datos como estos.

## Paso 3: El esquema SQL


```sql
-- Archivo: init-db/schema.sql
CREATE TABLE IF NOT EXISTS ventas_limpias (
    id SERIAL PRIMARY KEY,
    prodeucto VARCHAR(100) NOT NULL,
    cantidad INTEGER NOT NULL CHECK (cantidad > 0),
    precio_unitario DECIMAL(10, 2) NOT NULL,
    total DECIMAL(10, 2) NOT NULL,
    fecha DATE NOT NULL,
    cliente VARCHAR(100),
    ciudad VARCHAR(50) NOT NULL,
    cargado_en TIMESTAMP DEFAULT NOW()
);
```


- `CREATE TABLE IF NOT EXISTS ventas_limpias` → Tabla destino para datos ya validados.
- `id SERIAL PRIMARY KEY` → ID autoincremental gestionado por PostgreSQL.
- `cantidad INTEGER NOT NULL CHECK (cantidad > 0)` → Restricción a nivel de DB: rechaza cantidades ≤ 0.
- `total DECIMAL(10,2) NOT NULL` → Campo calculado (cantidad × precio). Hasta 10 dígitos, 2 decimales.
- `cargado_en TIMESTAMP DEFAULT NOW()` → Auditoría: cuándo se insertó cada fila.

Las restricciones CHECK son tu última línea de defensa: si el ETL tiene un bug, la DB rechaza datos inválidos.

## Paso 4: El script ETL en Python


- `esperar_postgres()` → Retry loop que intenta conectar cada 2 segundos. Necesario porque depends_on solo espera que el contenedor arranque, no que PostgreSQL esté listo para queries.
- `leer_csv()` → EXTRACT: lee el archivo con csv.DictReader (cada fila es un diccionario).
- `limpiar_fila()` → TRANSFORM: valida cantidad (> 0), normaliza fechas (soporta 2 formatos), asigna "Desconocido" a clientes vacíos, calcula el total.
- `cargar_en_postgres()` → LOAD: inserta filas limpias con queries parametrizadas (%s). Hace commit al final.
- `os.getenv("DB_HOST", "postgres")` → Lee la config desde variables de entorno (best practice en Docker). Si no existe, usa "postgres" por defecto.

## Paso 5: Dockerfile para el ETL


```Dockerfile
# Archivo: Dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY etl/ ./etl/
COPY data/ ./data/

CMD ["python", "etl/pipeline.py"]
```

 `FROM python:3.12-slim` → Base ligera con Python 3.12.
- `WORKDIR /app` → Todo ocurre desde /app.
- `COPY requirements.txt .` y `RUN pip install...` → Instala dependencias primero (cache).
- `COPY etl/ ./etl/` → Copia el directorio con el script.
- `COPY data/ ./data/` → Copia los CSVs de entrada.
- `CMD ["python", "etl/pipeline.py"]` → Al arrancar el contenedor, ejecuta el pipeline.

## Paso 6: El docker-compose.yml que une todo


```yml

# Archivo: docker-compose.yml
services:
  postgres:
    image: postgres:16
    environment:
      POSTGRES_USER: etl_user
      POSTGRES_PASSWORD: etl_secreto
      POSTGRES_DB: ventas
    ports:
      - "5432:5432"
    volumes:
      - pgdata:/var/lib/postgressql/data
      - ./init-db:/docker-entrypoint-initdb.d # Ejecuta schema.sql al inicio
    
  pgadmin:
    image: dpage/pgadmin4:latest
    environment:
      PGADMIN_DEFAULT_EMAIL: admin@datos.com
      PGADMIN_DEFAULT_PASSWORD: admin123
    ports:
      - "8080:80"
    depends_on:
      - postgres

  etl:
    build: .
    environment:
      DB_HOST: postgres
      DB_USER: etl_user
      DB_PASS: etl_secreto
      DB_NAME: ventas
    depends_on:
      - postgres
    volumes:
      - ./data:/app/data # Montar los CSVs

volumes:
  pgdata:
```

- `postgres:` → DB con volumen persistente + script init que crea la tabla.
- `./init-db:/docker-entrypoint-initdb.d` → Monta la carpeta entera. Cualquier .sql ahí se ejecuta al primer arranque.
- `pgadmin:` → Interfaz web para verificar datos visualmente (localhost:8080).
- `etl:` con `build: .` → Construye la imagen desde el Dockerfile local.
- `DB_HOST: postgres` → Dentro de compose, el hostname de la DB es "postgres" (nombre del servicio).
- `depends_on: - postgres` → ETL arranca después de que PostgreSQL esté levantado.
- `./data:/app/data` → Bind mount: puedes cambiar el CSV sin reconstruir la imagen.
